# Log Analysis

This notebook parses TensorBoard event logs under `tensorlogs/` and evaluation results under `logs/` to visualize training curves and summarize test outcomes.

_Run the cells below after launching the notebook from the repository root so that relative paths resolve correctly._


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from tensorboard.backend.event_processing import event_accumulator

plt.rcParams["figure.figsize"] = (8, 4)


In [2]:
repo_root = Path.cwd().parent
tensorlogs_dir = repo_root / "tensorlogs"
logs_dir = repo_root / "logs"

print(f"TensorBoard runs: {sorted(p.name for p in tensorlogs_dir.glob('*') if p.is_dir())}")
print(f"Evaluation runs: {sorted(p.name for p in logs_dir.glob('*') if p.is_dir())}")


TensorBoard runs: ['2025-11-02_22-57-28_BC_PP']
Evaluation runs: ['2025-11-02-22-45-40-test-pp-a2', '2025-11-02-22-47-09-test-grasp-a2', '2025-11-02-22-57-28-train-a2', '2025-11-02-23-07-20-test-grasp-a2-unseen', '2025-11-02-23-13-58-test-pp-a2-unseen']


## TensorBoard Scalars

The helper below loads scalar histories from each event file under `tensorlogs/` into a single DataFrame.


In [3]:
def load_tensorboard_scalars(tensorlogs_dir):
    records = []
    if not tensorlogs_dir.exists():
        print(f"Directory not found: {tensorlogs_dir}")
        return pd.DataFrame(records)
    for run_path in sorted(tensorlogs_dir.iterdir()):
        if not run_path.is_dir():
            continue
        event_files = sorted(run_path.glob("events.out.tfevents.*"))
        if not event_files:
            print(f"No event files in {run_path.name}")
            continue
        for event_path in event_files:
            try:
                ea = event_accumulator.EventAccumulator(str(event_path), size_guidance={"scalars": 0})
                ea.Reload()
            except Exception as exc:
                print(f"Could not load {event_path.name}: {exc}")
                continue
            for tag in ea.Tags().get("scalars", []):
                for event in ea.Scalars(tag):
                    records.append({
                        "run": run_path.name,
                        "event_file": event_path.name,
                        "tag": tag,
                        "step": event.step,
                        "wall_time": event.wall_time,
                        "value": event.value,
                    })
    df = pd.DataFrame.from_records(records)
    if df.empty:
        return df
    return df.sort_values(["tag", "run", "step"]).reset_index(drop=True)


In [4]:
tb_scalars = load_tensorboard_scalars(tensorlogs_dir)
tb_scalars.head()


,run,event_file,tag,step,wall_time,value
0,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,0,1.762121e+09,3.155708
1,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,1,1.762121e+09,3.082009
2,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,2,1.762121e+09,3.029096
3,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,3,1.762121e+09,2.852364
4,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,4,1.762121e+09,2.723838


In [5]:
tb_scalars['tag'].unique()

array(['loss/epoch', 'loss/iteration'], dtype=object)

In [ ]:
sns.lineplot(
    tb

## Test Log Summaries

The utilities below read per-case result files written during evaluation runs under `logs/`.


In [6]:
METRIC_LABELS = {
    4: ["avg_success", "avg_step", "avg_success_step", "avg_reward"],
    5: ["avg_success", "avg_grasp_success", "avg_place_success", "avg_step", "avg_success_step"],
}

def detect_run_type(run_name):
    lowered = run_name.lower()
    if "train" in lowered:
        return "train"
    if "grasp" in lowered:
        return "grasp"
    if "pp" in lowered:
        return "pick-place"
    if "place" in lowered:
        return "place"
    return "unknown"

def parse_case_result(case_path):
    text = case_path.read_text().strip()
    if not text:
        return None
    tokens = text.split()
    float_positions = []
    float_values = []
    for idx, token in enumerate(tokens):
        try:
            value = float(token)
        except ValueError:
            continue
        float_positions.append(idx)
        float_values.append(value)
    if not float_values:
        return None
    first_value_pos = float_positions[0]
    language_tokens = tokens[:first_value_pos]
    language_goal = " ".join(language_tokens)
    return language_goal, float_values

def load_test_results(logs_dir):
    rows = []
    if not logs_dir.exists():
        print(f"Directory not found: {logs_dir}")
        return pd.DataFrame()
    for run_path in sorted(logs_dir.iterdir()):
        if not run_path.is_dir():
            continue
        results_dir = run_path / "results"
        if not results_dir.exists():
            continue
        for case_path in sorted(results_dir.glob("case*.txt")):
            parsed = parse_case_result(case_path)
            if not parsed:
                continue
            language_goal, metrics = parsed
            labels = METRIC_LABELS.get(len(metrics), [f"metric_{i}" for i in range(len(metrics))])
            row = {
                "run": run_path.name,
                "run_type": detect_run_type(run_path.name),
                "case": case_path.stem,
                "language_goal": language_goal,
            }
            for label, value in zip(labels, metrics):
                row[label] = value
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["run", "case"]).reset_index(drop=True)


In [7]:
test_results = load_test_results(logs_dir)
test_results.head()


,run,run_type,case,language_goal,avg_success,avg_grasp_success,avg_place_success,avg_step,avg_success_step,avg_reward
0,2025-11-02-22-45-40-test-pp-a2,pick-place,case0,I want a strawberry put it near the apple,0.466667,0.466667,0.466667,6.933333,5.714286,NaN
1,2025-11-02-22-45-40-test-pp-a2,pick-place,case1,I need a cup put it around the box,1.000000,1.000000,1.000000,1.133333,1.133333,NaN
2,2025-11-02-22-45-40-test-pp-a2,pick-place,case2,I need the box place this around the toy,0.800000,0.800000,0.800000,3.533333,2.416667,NaN
3,2025-11-02-22-45-40-test-pp-a2,pick-place,case3,grasp the red fruit move the object in the red...,0.800000,1.000000,0.800000,3.333333,3.250000,NaN
4,2025-11-02-22-45-40-test-pp-a2,pick-place,case4,give me a cup move the object near the racquet...,0.133333,0.866667,0.133333,2.200000,4.000000,NaN


In [8]:

metric_columns = [col for col in test_results.columns if col.startswith("avg_")]
summary = (
    test_results
    .groupby(["run", "run_type"])[metric_columns]
    .mean()
    .reset_index()
    .sort_values("avg_success", ascending=False)
)
display(summary)


,run,run_type,avg_success,avg_grasp_success,avg_place_success,avg_step,avg_success_step,avg_reward
1,2025-11-02-22-47-09-test-grasp-a2,grasp,0.946667,NaN,NaN,2.080000,2.035556,0.728586
2,2025-11-02-23-07-20-test-grasp-a2-unseen,grasp,0.933333,NaN,NaN,2.773333,2.515018,0.457278
0,2025-11-02-22-45-40-test-pp-a2,pick-place,0.758333,0.875,0.758333,2.766667,2.580311,NaN
3,2025-11-02-23-13-58-test-pp-a2-unseen,pick-place,0.400000,0.750,0.400000,3.533333,252.851190,NaN


In [9]:
if "avg_success" in test_results.columns:
    top_cases = test_results.sort_values("avg_success", ascending=False)
    display(top_cases.head(10))


,run,run_type,case,language_goal,avg_success,avg_grasp_success,avg_place_success,avg_step,avg_success_step,avg_reward
1,2025-11-02-22-45-40-test-pp-a2,pick-place,case1,I need a cup put it around the box,1.0,1.0,1.0,1.133333,1.133333,NaN
9,2025-11-02-22-47-09-test-grasp-a2,grasp,case1,get something to eat,1.0,NaN,NaN,1.800000,1.800000,0.907900
6,2025-11-02-22-45-40-test-pp-a2,pick-place,case6,I want a round object move the object in the r...,1.0,1.0,1.0,1.266667,1.266667,NaN
8,2025-11-02-22-47-09-test-grasp-a2,grasp,case0,grasp a round object,1.0,NaN,NaN,2.200000,2.200000,0.811073
7,2025-11-02-22-45-40-test-pp-a2,pick-place,case7,get something to eat move the object next to t...,1.0,1.0,1.0,1.400000,1.400000,NaN
20,2025-11-02-23-07-20-test-grasp-a2-unseen,grasp,case2,I need a suger,1.0,NaN,NaN,2.000000,2.000000,0.702825
17,2025-11-02-22-47-09-test-grasp-a2,grasp,case9,give me the thera_med,1.0,NaN,NaN,3.200000,3.200000,0.506316
15,2025-11-02-22-47-09-test-grasp-a2,grasp,case7,give me the pear,1.0,NaN,NaN,1.000000,1.000000,1.000000
14,2025-11-02-22-47-09-test-grasp-a2,grasp,case6,I need a fruit,1.0,NaN,NaN,1.400000,1.400000,0.941717
13,2025-11-02-22-47-09-test-grasp-a2,grasp,case5,I need a cup,1.0,NaN,NaN,1.000000,1.000000,1.000000
